# Hubble-8B unlearning — guided hyperparameter sweep

For each method, sweeps a small grid on the 1–2 key knobs (`hubble8b/sweep.py::SWEEP_GRIDS`) and, via in-loop evaluation, checkpoints every `EVAL_EVERY` epochs — so `epochs` is swept for free. Selection metric: `combined_score = retain_acc − λ·max(0, forget_acc − chance)` on a fast `full_prefix` infill proxy (Keep vs Unlearn). The best-scoring checkpoint per method is pushed (public) with its full 4-format infill CSV, so `master.ipynb` can aggregate it. Runs one method at a time (onlygpu1 QOS).

In [ ]:
# --- Setup: install deps, put the hubble8b package on the path, set HF token ---
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.53", "datasets", "accelerate", "bitsandbytes", "scipy"], check=False)

HUB_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
for cand in [HUB_DIR, os.path.join(HUB_DIR, "gpu_8b_run"), "/kaggle/input"]:
    if os.path.isdir(os.path.join(cand, "hubble8b")):
        sys.path.insert(0, cand)
        break

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is not set. Export HF_TOKEN before submitting the job.")

import hubble8b as h8
h8.hf_login()
print("device:", h8.DEVICE, "| target slot:", h8.TARGET_SLOT, "| LSR slot:", h8.LSR_SLOT)

In [ ]:
# --- Data + fast proxy evaluator (forget/retain infill on full_prefix) ---
forget_set, keep_set, unseen_set = h8.load_yago_splits()
wiki = h8.load_retain_texts()
splits = {"Unseen": unseen_set, "Unlearn": forget_set, "Keep": keep_set}
pool_birth = h8.build_distractor_pool(forget_set, keep_set, unseen_set, h8.TARGET_SLOT)
tok = h8.get_tokenizer()
eval_fn = h8.make_eval_fn(forget_set, keep_set, pool_birth, tok)

# --- Retain training data: KEEP biographies (the actual retain set D_r) + WikiText fluency ---
# The earlier collapse came from retaining on WikiText only, which never protects the Keep
# entities. Standard unlearning uses D_r in the loss, so we oversample the 45 Keep biographies
# to ~50% of the pool and mix in general WikiText for the other ~50%.
n = len(forget_set) * 40
keep_texts = list(keep_set["text"])
reps = max(1, (n // 2) // max(1, len(keep_texts)))
wiki_texts = list(wiki.select(range(min(len(wiki), n // 2)))["text"])
retain_texts = keep_texts * reps + wiki_texts
print(f"retain pool: {len(keep_texts)} keep bios x{reps} + {len(wiki_texts)} wiki = {len(retain_texts)}")

# --- Sweep knobs ---
# Override the method list per job with env SWEEP_METHODS (e.g. "npo" or "npo,rmu") so you can
# split the sweep into short, backfill-friendly per-method jobs. Default: all four (serial).
METHODS_TO_SWEEP = [m.strip() for m in os.environ.get("SWEEP_METHODS", "npo,graddiff,idk,rmu").split(",") if m.strip()]
EVAL_EVERY   = int(os.environ.get("SWEEP_EVAL_EVERY", "2"))   # eval+checkpoint every N epochs -> free `epochs`
SWEEP_EPOCHS = int(os.environ.get("SWEEP_EPOCHS", "8"))       # upper bound; best-checkpoint picks the stop
PUSH_WINNERS = os.environ.get("SWEEP_PUSH", "1") != "0"       # push winner (public) + full infill CSV

print("sweeping:", METHODS_TO_SWEEP, "| eval_every:", EVAL_EVERY, "| epochs:", SWEEP_EPOCHS, "| push:", PUSH_WINNERS)
print("score: lambda:", h8.SWEEP_LAMBDA, "| forget_thresh:", h8.SWEEP_FORGET_THRESH)
for m in METHODS_TO_SWEEP:
    print(f"  {m}: {h8.SWEEP_GRIDS[m]}")

## Run the sweep — per method: grid → best checkpoint → (optional) push winner + full infill CSV

In [ ]:
import json, pandas as pd
DISPLAY = {"npo": "NPO", "graddiff": "GradDiff", "idk": "IDK", "rmu": "RMU"}
all_results, winners_hp, summary = [], {}, []

for method in METHODS_TO_SWEEP:
    df, best = h8.run_sweep(method, forget_set, keep_set, retain_texts, eval_fn,
                            eval_every=EVAL_EVERY, epochs=SWEEP_EPOCHS)
    if len(df):
        all_results.append(df)
    if best is None:
        print(f"[{method}] produced no winner; skipping"); continue
    cfg = best["config"]
    winners_hp[method] = dict(cfg, epochs=best["epoch"])   # merge into config.HP to reproduce
    summary.append(dict(method=method, **cfg, best_epoch=best["epoch"], score=round(best["score"], 4),
                        forget_acc=round(best["forget_acc"], 4), retain_acc=round(best["retain_acc"], 4)))
    if PUSH_WINNERS:
        name = DISPLAY[method]; m = h8.to_gpu(best["model"])
        inf = h8.run_infill(m, name, splits, pool_birth, target_slot=h8.TARGET_SLOT)  # full 4-format
        ip = f"{h8.workdir()}/infill_{name}.csv"; inf.to_csv(ip, index=False); h8.upload_artifact(ip)
        h8.push_model(m, tok, method)                      # public model repo
        h8.to_cpu(m); del m
    best["model"] = None; h8.free_cuda()                    # free the winner before the next method

if all_results:
    sweep_df = pd.concat(all_results, ignore_index=True)
    p = f"{h8.workdir()}/sweep_results.csv"; sweep_df.to_csv(p, index=False); h8.upload_artifact(p)
    print("\nsaved", p, "| rows:", len(sweep_df))
hp_path = f"{h8.workdir()}/winners_hp.json"
json.dump(winners_hp, open(hp_path, "w"), indent=2); h8.upload_artifact(hp_path)

print("\n=== SWEEP WINNERS ===")
print(pd.DataFrame(summary).to_string(index=False) if summary else "(none)")
print("\nMerge these into hubble8b/config.py HP for reproducible runs:")
for method, hp in winners_hp.items():
    print(f"  {method!r}: " + ", ".join(f"{k}={v}" for k, v in hp.items()))

After this completes, the winners are on HF (public) with their infill CSVs and `sweep_results.csv` / `winners_hp.json` in the results dataset. Update `config.HP` with the printed values, then run `run_master.sbatch` to produce the figures (aggregate bars, forget-vs-retain logit lens, LSR).